# Oxford-IIIT Pet ResNet-50 training

This notebook trains the ResNet-50 production candidate for the Oxford-IIIT Pet breed classifier. It is intentionally reproducible and restart-safe.

Rules enforced here:
- Use the existing DVC-generated manifest and committed 37-class label map; do not create a new split.
- Use clean `train` and `val` records only. Test and corrupted records are not loaded or scored in this notebook.
- Convert every image to RGB and use one shared evaluation transform.
- Fail loudly when data or pretrained weights are unavailable; never fabricate fallback data.
- Save model weights together with the label map and transform metadata required by serving.

Run this notebook from the project environment with `uv run jupyter lab`.

In [1]:
from __future__ import annotations

import json
import os
import random
import time
from collections.abc import Callable
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import f1_score
from torch import Tensor, nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from pet_breed_classification import cfg


PROJECT_ROOT = cfg.PROJECT_ROOT
MANIFEST_PATH = cfg.MANIFEST_PATH
LABEL_MAP_PATH = cfg.LABEL_MAP_PATH
SEED = cfg.SEED
NUM_WORKERS_CONFIG = cfg.NUM_WORKERS
IMAGE_SIZE = 224
RESIZE_SIZE = 256
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def to_rgb(image: Image.Image) -> Image.Image:
    return image.convert("RGB")


def build_train_transform() -> transforms.Compose:
    return transforms.Compose([
        transforms.Lambda(to_rgb),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def build_eval_transform() -> transforms.Compose:
    return transforms.Compose([
        transforms.Lambda(to_rgb),
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class PetBreedDataset(Dataset[tuple[Tensor, int]]):
    def __init__(
        self,
        records: list[dict],
        project_root: Path,
        transform: Callable[[Image.Image], Tensor],
    ) -> None:
        self.records = records
        self.project_root = project_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, int]:
        record = self.records[index]
        image_path = self.project_root / str(record["path"])
        try:
            with Image.open(image_path) as image:
                tensor = self.transform(image)
        except Exception as exc:
            raise RuntimeError(f"Could not load image {image_path}") from exc
        return tensor, int(record["class_index"])


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


class PetBreedClassifier(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = False) -> None:
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, images: Tensor) -> Tensor:
        return self.backbone(images)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: /home/tamimi6030/workspace/Pet-Breed-Classification
Device: cuda
GPU: NVIDIA GeForce RTX 3060


In [2]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print(f"Seed: {SEED}")

Seed: 42


## Load and validate the fixed data contract

The manifest contains clean images and a labeled corruption suite. Only clean train/validation rows are selected below. The official test split is deliberately excluded so test metrics remain untouched until the required final evaluation.

In [3]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Restore the DVC data artifacts before training."
    )
if not LABEL_MAP_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {LABEL_MAP_PATH}. Restore the committed label map before training."
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
label_map = json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))
classes = label_map.get("classes")
if not isinstance(classes, list):
    raise ValueError("Label map must contain a classes list")

assert len(classes) == 37, f"Expected 37 classes, found {len(classes)}"
assert [entry["index"] for entry in classes] == list(range(37))
class_to_idx = {entry["breed"]: entry["index"] for entry in classes}
assert len(class_to_idx) == 37

clean_train_val = [
    record
    for record in manifest
    if record["corruption"] is None and record["split"] in {"train", "val"}
]
assert clean_train_val, "No clean train/validation records were found"
assert all(record["split"] in {"train", "val"} for record in clean_train_val)
assert all(record["breed"] in class_to_idx for record in clean_train_val)
assert all(record["class_index"] == class_to_idx[record["breed"]] for record in clean_train_val)

for record in clean_train_val:
    image_path = PROJECT_ROOT / record["path"]
    if not image_path.is_file():
        raise FileNotFoundError(f"Manifest image does not exist: {image_path}")

train_records = [record for record in clean_train_val if record["split"] == "train"]
val_records = [record for record in clean_train_val if record["split"] == "val"]
print(f"Train images: {len(train_records):,}")
print(f"Validation images: {len(val_records):,}")
print(f"Classes: {len(class_to_idx)}")

Train images: 2,944
Validation images: 736
Classes: 37


In [4]:
train_dataset = PetBreedDataset(train_records, PROJECT_ROOT, build_train_transform())
val_dataset = PetBreedDataset(val_records, PROJECT_ROOT, build_eval_transform())

sample_image, sample_label = val_dataset[0]
assert sample_image.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert sample_image.dtype == torch.float32
print(f"Validation sample tensor: {tuple(sample_image.shape)}, label={sample_label}")

Validation sample tensor: (3, 224, 224), label=0


## Build deterministic data loaders

The worker seed and loader generator are explicit. The validation loader is never shuffled and uses the deterministic evaluation transform.

In [5]:
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 16
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
requested_workers = NUM_WORKERS_CONFIG if NUM_WORKERS_CONFIG is not None else max(1, (os.cpu_count() or 2) - 1)
NUM_WORKERS = min(requested_workers, 8)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)
loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
    "persistent_workers": NUM_WORKERS > 0,
}
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, **loader_options
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_options
)
print(f"Batch size: {BATCH_SIZE}; workers: {NUM_WORKERS}")

Batch size: 64; workers: 8


## Create the pretrained ResNet-50

The ImageNet backbone is fine-tuned end to end. The classification head is replaced with exactly 37 outputs. Downloading pretrained weights is expected to fail loudly if the environment has no access to the weights.

In [6]:
model = PetBreedClassifier(num_classes=len(class_to_idx), pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}")
print(f"Automatic mixed precision: {use_amp}")

Trainable parameters: 23,583,845
Automatic mixed precision: True


In [7]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    predictions: list[int] = []
    targets_seen: list[int] = []

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            batch_size = targets.size(0)
            batch_predictions = logits.argmax(dim=1)
            total_loss += loss.item() * batch_size
            total_correct += (batch_predictions == targets).sum().item()
            total_examples += batch_size
            predictions.extend(batch_predictions.detach().cpu().tolist())
            targets_seen.extend(targets.detach().cpu().tolist())

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
        "macro_f1": f1_score(
            targets_seen, predictions, average="macro", zero_division=0
        ),
    }

## Train and save the best validation checkpoint

The checkpoint is selected using validation accuracy only. No test image is read. The saved metadata is JSON-compatible and contains the exact class mapping and deterministic serving transform.

In [8]:
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACT_DIR / "resnet50_best.pt"
HISTORY_PATH = ARTIFACT_DIR / "resnet50_training_history.json"

idx_to_class = {str(entry["index"]): entry["breed"] for entry in classes}
eval_transform_metadata = {
    "color_mode": "RGB",
    "resize_size": RESIZE_SIZE,
    "crop_size": IMAGE_SIZE,
    "mean": list(IMAGENET_MEAN),
    "std": list(IMAGENET_STD),
}


def checkpoint_payload(epoch: int, metrics: dict[str, float]) -> dict:
    return {
        "model_name": "resnet50",
        "backbone": "resnet50",
        "num_classes": len(class_to_idx),
        "classes": classes,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "eval_transform": eval_transform_metadata,
        "seed": SEED,
        "temperature": 1.0,
        "abstain_threshold": None,
        "epoch": epoch,
        "validation_metrics": metrics,
        "model_state_dict": {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        },
    }


history: list[dict[str, float]] = []
best_val_accuracy = -1.0
best_epoch = 0
training_started_at = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    started_at = time.perf_counter()
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step()
    elapsed_seconds = time.perf_counter() - started_at

    row = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "elapsed_seconds": elapsed_seconds,
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={row['train_loss']:.4f}, acc={row['train_accuracy']:.4f} | "
        f"val loss={row['val_loss']:.4f}, acc={row['val_accuracy']:.4f}, "
        f"macro-F1={row['val_macro_f1']:.4f} | {elapsed_seconds:.1f}s"
    )

    if row["val_accuracy"] > best_val_accuracy:
        best_val_accuracy = row["val_accuracy"]
        best_epoch = epoch
        torch.save(checkpoint_payload(epoch, val_metrics), CHECKPOINT_PATH)
        print(f"Saved new best checkpoint: {CHECKPOINT_PATH}")

HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")
total_training_seconds = time.perf_counter() - training_started_at
print(f"Best epoch: {best_epoch}; validation accuracy: {best_val_accuracy:.4f}")
print(f"Total training time: {total_training_seconds:.1f}s")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"History: {HISTORY_PATH}")

Epoch 01/10 | train loss=2.6459, acc=0.5537 | val loss=0.6405, acc=0.8832, macro-F1=0.8798 | 11.0s
Saved new best checkpoint: /home/tamimi6030/workspace/Pet-Breed-Classification/artifacts/resnet50_best.pt
Epoch 02/10 | train loss=0.5205, acc=0.9198 | val loss=0.3146, acc=0.9416, macro-F1=0.9415 | 10.1s
Saved new best checkpoint: /home/tamimi6030/workspace/Pet-Breed-Classification/artifacts/resnet50_best.pt
Epoch 03/10 | train loss=0.1553, acc=0.9674 | val loss=0.2414, acc=0.9375, macro-F1=0.9381 | 10.3s
Epoch 04/10 | train loss=0.0782, acc=0.9857 | val loss=0.2214, acc=0.9375, macro-F1=0.9376 | 10.0s
Epoch 05/10 | train loss=0.0458, acc=0.9935 | val loss=0.2096, acc=0.9361, macro-F1=0.9367 | 10.0s
Epoch 06/10 | train loss=0.0310, acc=0.9959 | val loss=0.2130, acc=0.9361, macro-F1=0.9368 | 10.0s
Epoch 07/10 | train loss=0.0236, acc=0.9980 | val loss=0.2032, acc=0.9361, macro-F1=0.9367 | 9.9s
Epoch 08/10 | train loss=0.0194, acc=0.9986 | val loss=0.1968, acc=0.9402, macro-F1=0.9407 | 9.8

In [9]:
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except Exception as exc:
    print(f"Safe checkpoint loading failed: {type(exc).__name__}: {exc}")
    raise
assert checkpoint["model_name"] == "resnet50"
assert checkpoint["backbone"] == "resnet50"
assert checkpoint["class_to_idx"] == class_to_idx
assert checkpoint["classes"] == classes
assert len(checkpoint["classes"]) == checkpoint["num_classes"]
assert checkpoint["eval_transform"] == eval_transform_metadata
assert checkpoint["num_classes"] == 37
assert checkpoint["temperature"] == 1.0
assert checkpoint["abstain_threshold"] is None
print("Checkpoint validation passed.")
print("Test metrics were intentionally not computed in this notebook.")

Checkpoint validation passed.
Test metrics were intentionally not computed in this notebook.
